In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.4 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

In [ ]:
!pip install transformers torch gradio deep-translator accelerate -q

In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from deep_translator import GoogleTranslator

# ─────────────────────────────────────────────
# 1. Load TinyLlama 1.1B Chat
#    ✅ Fully open — no HuggingFace login needed
# ─────────────────────────────────────────────
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer and model…")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Model loaded successfully!")

# ─────────────────────────────────────────────
# 2. Core processing function
# ─────────────────────────────────────────────
def process_query(bengali_input):
    try:
        # ── Step A: Bengali → English ──────────────────────────────────
        english_query = GoogleTranslator(source='bn', target='en').translate(bengali_input)

        # ── Step B: Build TinyLlama chat prompt & generate ─────────────
        # TinyLlama uses the ChatML format:
        # <|system|>...</s><|user|>...</s><|assistant|>
        messages = [
            {
                "role": "system",
                "content": "You are a helpful, accurate, and concise assistant. Answer the user's question clearly."
            },
            {
                "role": "user",
                "content": english_query
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_length = inputs["input_ids"].shape[1]

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens (skip the prompt)
        generated_tokens = outputs[0][input_length:]
        english_response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # ── Step C: English → Bengali ──────────────────────────────────
        bengali_response = GoogleTranslator(source='en', target='bn').translate(english_response)

        return english_query, english_response, bengali_response

    except Exception as e:
        err = f"Error: {str(e)}"
        return err, err, err

# ─────────────────────────────────────────────
# 3. Gradio Interface
# ─────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🦙 Bengali ↔️ TinyLlama AI Interface")
    gr.Markdown(
        "Type a question in **Bengali**. "
        "It is translated to English for TinyLlama, "
        "then the answer is translated back to Bengali."
    )

    with gr.Row():
        with gr.Column():
            bn_in = gr.Textbox(
                label="Bengali Input (আপনার প্রশ্ন)",
                placeholder="এখানে লিখুন…",
                lines=4
            )
            submit_btn = gr.Button("Submit / পাঠান", variant="primary")

        with gr.Column():
            bn_out = gr.Textbox(
                label="Bengali Output (উত্তর)",
                interactive=False,
                lines=5
            )

    with gr.Accordion("🔍 View Translation Pipeline (Under the hood)", open=False):
        en_in  = gr.Textbox(label="1. Translated English Query",      interactive=False)
        en_out = gr.Textbox(label="2. TinyLlama's English Response",  interactive=False)

    submit_btn.click(
        fn=process_query,
        inputs=bn_in,
        outputs=[en_in, en_out, bn_out]
    )

demo.launch(debug=True)

Loading tokenizer and model…


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded successfully!


/tmp/ipykernel_14504/2058174265.py:80: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39d3a36c7a3f3f91d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://39d3a36c7a3f3f91d9.gradio.live
